# NSGA-II (geatpy) multi-objective optimization: Maximizing risk improvement + minimizing economic cost (based on risk probability after GWR raw output mapped by robust sigmoid)

This Notebook uses **geatpy's NSGA-II** (`moea_NSGA2_templet`) to do **dual-objective optimization** on the **GWR original regression output and the risk probability** after mapping the robust sigmoid parameters saved in the training stage.

## Two objective functions
- **Objective 1: Relative risk improvement rate (the higher, the better)**
  Record the baseline risk probability as \(y_{base}\), and the optimized risk probability as \(y_{opt}\), defined
  \[
  RR = \frac{y_{base}-y_{opt}}{\max(y_{base}, \varepsilon)}
  \]  
  Among them, \(\varepsilon\) is a very small number, avoid division by zero. ZXCVBNM0 A larger ZXCVBNM indicates a greater risk reduction (a more significant improvement).

> Here `y_base` / `y_opt` is not the result of simple `clip(0,1)`.
> They uniformly use `transform_metadata` saved in `gwr_classification_artifacts.pkl`,
> Obtained by doing the same set of robust sigmoid mapping on the original output of GWR.

- **Goal 2: Total economic cost (the lower, the better)**
  \[
  C_{total}=C_{ISA}+C_{WT}+C_{LAI}
  \]  
  (),.

## Decision variables (only 3)
- `IP`: Impervious Surface Area / ImperviousIndex (0–100, floating up and down according to the value of this row; only allowed to be reduced by up to 30%)
- `LAI`: Leaf Area Index (according to the value of this row ±30%)
- `WTD`: Water Table Depth (according to the value of this row ±30%)

7 ,,.

---

## Operation mode
Still use **comment/uncomment** to select the point to optimize (single point/topN/interval) and then run NSGA-II. The final output will be:
- `summary`: Select a "compromise solution (minimum knee/ideal distance)" result for each point
- `pareto`: Pareto front solution set for each point (long table)


In [ ]:
# =========================
# 1) Environment and path settings (run first)
# =========================
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

import geatpy as ea

# --- :mgtwr () ---
# Linux / macOS ():
sys.path.append("/path/to/sinkhole-risk-china/code")

# Windows Alternate (if you are running on Windows, replace the above with this path):
# sys.path.append(r"D:\path\to\sinkhole-risk-china\code") 

# mgtwr import()
from mgtwr.sel_bw import Sel_BW
from mgtwr.model import Model
import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils

print("Imports OK. mgtwr and gwr_sigmoid_utils are available.")


In [ ]:
ssp = "ssp2"  # TODO: 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2080"  # TODO: '2020' / '2040' / '2060' / '2080' / '2100'

resolution = "10km"  # TODO: Manually change the resolution to '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km' as needed
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)
# ✅ Output directory: base_path/outputs/path_name
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time, "optimization")
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# ==================================
# 2) File path & running parameters (modify as needed)
# ==================================

# Your CSV (contains Longitude, Latitude and 10 feature columns)
# CSV_PATH = os.path.join(data_base_path, "Extracted_HAVE_future", f"Points_China_all_{resolution}", f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned_optimization.csv")
CSV_PATH = os.path.join(
    data_base_path,
    "Extracted_HAVE_future",
    f"Points_China_all_{resolution}",
    f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned.csv"
)

# pkl( key='gwr')
MODEL_PKL_PATH = os.path.join(base_path, "code", "3_gwr_model_train", "national", "GWR", "gwr_model_national_GWR.pkl")
CLASSIFICATION_ARTIFACTS_PATH = os.path.join(
    base_path,
    "code",
    "3_gwr_model_train",
    "national",
    "GWR",
    "gwr_classification_artifacts.pkl",
)
CLASS_BOUNDARIES_PATH = os.path.join(
    base_path,
    "code",
    "3_gwr_model_train",
    "national",
    "GWR",
    "class_boundaries.pkl",
)
# -------- NSGA-II ()--------
# Speed ​​priority default: The current GWR prediction cost is very high, and it is switched to fast by default to avoid the extremely large search scale of 60x80.
# Optional: "balanced" / "fast" / "ultra_fast"
SPEED_PRESET = "fast"

if SPEED_PRESET == "balanced":
    NIND = 36
    MAXGEN = 30
    PM = 0.18
    XOVER = 0.75
    RANDOM_SAMPLE_FRAC_DEFAULT = 0.10
    SAVE_EVERY_ROWS_DEFAULT = 50
elif SPEED_PRESET == "ultra_fast":
    NIND = 16
    MAXGEN = 10
    PM = 0.12
    XOVER = 0.70
    RANDOM_SAMPLE_FRAC_DEFAULT = 0.10
    SAVE_EVERY_ROWS_DEFAULT = 150
else:
    SPEED_PRESET = "fast"
    NIND = 24
    MAXGEN = 16
    PM = 0.15
    XOVER = 0.72
    RANDOM_SAMPLE_FRAC_DEFAULT = 0.10
    SAVE_EVERY_ROWS_DEFAULT = 100

SEED = 1    # ( row SEED + row_id,)

print(f"Config set. SPEED_PRESET = {SPEED_PRESET}")
print(f"Estimated search load per row ~= NIND x MAXGEN = {NIND * MAXGEN}")
print("Random sample ratio is fixed at 10% by requirement.")

# ==============================
# Cost model settings (2040-2060, default is 20 years of accumulation)
# ==============================
T_YEARS = 20.0          # 2040-2060:20

def infer_cell_area_km2(resolution_str: str) -> float:
    """Automatically calculate the grid area km^2 based on resolution (such as 10km / 5km / 0.1km)."""
    s = str(resolution_str).strip().lower()
    if not s.endswith("km"):
        raise ValueError(f"Unable to retrieve resolution={resolution_str!r}Automatically identify the grid side length, please set A_CELL_KM2 manually.")
    side_km = float(s[:-2])
    return side_km ** 2

# Key: Automatically infer the grid area from resolution to avoid using 1000 km^2 for 10km data
A_CELL_KM2 = infer_cell_area_km2(resolution)   # For example 10km -> 100 km^2
A_M2 = float(A_CELL_KM2) * 1e6
A_HA = A_M2 / 1e4
print(f"A_CELL_KM2 (auto-inferred from resolution={resolution}): {A_CELL_KM2} km^2")

# Processing step.
COST_MODE = "incremental"   # "absolute" or "incremental"
COST_REPORT = "total"       # "total"(20) "annual_avg"(=total/T_YEARS)
RISK_SPACE = "sigmoid_prob"

# -------- Cost balance parameters (control the three types of governance costs at the same order of magnitude) --------
# bal11: Use bal9's search space to maintain level 2/3 reduction capabilities, but switch to a more balanced final pick score.
# The goal is to keep the average cost-to-average ratio of the three categories low while approaching the bal9 level effect.
BALANCE_TAG = "bal11"
IP_MAX_REDUCTION_FRAC = 1.00
LAI_REL_CHANGE_FRAC = 0.80
WTD_REL_CHANGE_FRAC = 1.00
ISA_NET_COST_CAP_USD = 2.0e8
LAI_NET_COST_CAP_USD = 2.5e7
WTD_NET_COST_CAP_USD = 5.0e7

# Depaved percentage points, ISA ,
# At the same time, the penalty for LAI is slightly raised to avoid continued significant imbalance in the three types of costs.
INTERNAL_COST_WEIGHT_ISA = 0.03
INTERNAL_COST_WEIGHT_WT = 0.50
INTERNAL_COST_WEIGHT_LAI = 1.40
INTERNAL_BALANCE_PENALTY_WEIGHT = 3.00
RISK_LEVEL_DROP_OBJECTIVE_WEIGHT = 4.00
RISK_LEVEL_2PLUS_OBJECTIVE_BONUS = 2.00
RISK_LEVEL_3PLUS_OBJECTIVE_BONUS = 3.00

# When finally selecting a compromise solution from the Pareto solution, add a layer of preference of "the itemized costs should not differ too much".
KNEE_RR_WEIGHT = 1.00
KNEE_COST_WEIGHT = 0.50
KNEE_DROP_WEIGHT = 6.00
KNEE_BALANCE_LOG_WEIGHT = 2.00
KNEE_BALANCE_SHARE_WEIGHT = 4.00
KNEE_ISA_SHARE_WEIGHT = 0.00
KNEE_LAI_SHARE_EXCESS_WEIGHT = 0.00
KNEE_CAP_EXCESS_WEIGHT = 100.0
KNEE_MIN_RR_FRACTION = 0.00

# -------- ( 9 )--------
RANDOM_SAMPLE_ENABLED = True
RANDOM_SAMPLE_FRAC = RANDOM_SAMPLE_FRAC_DEFAULT
RANDOM_SAMPLE_SEED = 42
if RANDOM_SAMPLE_ENABLED:
    if not (0.0 < float(RANDOM_SAMPLE_FRAC) <= 1.0):
        raise ValueError("RANDOM_SAMPLE_FRAC must be in the range (0, 1].")
    SAMPLE_TAG = f"sample{int(round(float(RANDOM_SAMPLE_FRAC) * 100)):02d}pct_seed{int(RANDOM_SAMPLE_SEED)}"
else:
    SAMPLE_TAG = "sample100pct"
SAMPLE_COST_WEIGHT = 1.0  # n_full / n_sample

# -------- / ()--------
# Note:
# - checkpoint / final filename now only retains necessary information: population size, evolution generation, cost caliber, sampling label, balanced version.
# - checkpoint , BALANCE_TAG .
# - , RESUME_FROM_CHECKPOINT False, checkpoint .
# - SAVE_EVERY_ROWS It is recommended not to be too small, otherwise it will cause frequent hard disk writes.
RESUME_FROM_CHECKPOINT = True
SAVE_EVERY_ROWS = SAVE_EVERY_ROWS_DEFAULT
FSYNC_EVERY_SAVE = False

mode_tag = "inc" if str(COST_MODE).lower() == "incremental" else "abs"
report_tag = "tot" if str(COST_REPORT).lower() == "total" else "avg"
sample_tag = SAMPLE_TAG.replace("sample", "s").replace("pct_seed", "_r")
run_tag = f"{SPEED_PRESET}_n{NIND}_g{MAXGEN}_{mode_tag}_{report_tag}_{sample_tag}_{BALANCE_TAG}"
run_tag = run_tag.replace("/", "_").replace(" ", "_")

CHECKPOINT_SUMMARY_PATH = os.path.join(out_dir, f"nsga2_{run_tag}_summary_ckpt.csv")
CHECKPOINT_PARETO_PATH  = os.path.join(out_dir, f"nsga2_{run_tag}_pareto_ckpt.csv")
FINAL_SUMMARY_PATH = os.path.join(out_dir, f"nsga2_{run_tag}_summary.csv")
FINAL_PARETO_PATH = os.path.join(out_dir, f"nsga2_{run_tag}_pareto.csv")

# ---- Cost parameters (from the table you gave)----
# ISA
C_DEPAVE = 60.0     # USD/m^2 (one-time use)
C_FEE = 2.0         # USD/m^2/year ()
# WTD
C_ENERGY = 0.01     # USD/m^3·m
C_CAPITAL = 0.0005  # USD/m^3·m^2
# LAI
C_PLANT = 2328.0    # USD/ha (one-time use)
C_MAINT = 100.0     # USD/ha/year (year)

print(f"Run tag = {run_tag}")
print(f"Checkpoint summary: {CHECKPOINT_SUMMARY_PATH}")
print(f"Checkpoint pareto : {CHECKPOINT_PARETO_PATH}")
print(f"Risk space = {RISK_SPACE}")
print(f"Random sample enabled = {RANDOM_SAMPLE_ENABLED}")
print(f"Random sample frac = {RANDOM_SAMPLE_FRAC}")
print(f"Random sample seed = {RANDOM_SAMPLE_SEED}")
print(f"Checkpoint save every rows = {SAVE_EVERY_ROWS}")
print(f"Fsync every save = {FSYNC_EVERY_SAVE}")
print(f"Classification artifacts: {CLASSIFICATION_ARTIFACTS_PATH}")
print(f"Class boundaries: {CLASS_BOUNDARIES_PATH}")
print(f"Cost caps (20y total): ISA={ISA_NET_COST_CAP_USD:,.0f}, LAI={LAI_NET_COST_CAP_USD:,.0f}, WTD={WTD_NET_COST_CAP_USD:,.0f}")
print(f"Relative bounds: IP -{IP_MAX_REDUCTION_FRAC:.0%}, LAI ±{LAI_REL_CHANGE_FRAC:.0%}, WTD ±{WTD_REL_CHANGE_FRAC:.0%}")
ISA_NET_COST_PER_POINT_USD = max(float(C_DEPAVE) - float(T_YEARS) * float(C_FEE), 0.0) * (float(A_M2) / 100.0)
print(f"ISA net cost per 1 percentage-point reduction = {ISA_NET_COST_PER_POINT_USD:,.0f} USD")
print(
    f"Internal objective weights: ISA={INTERNAL_COST_WEIGHT_ISA:.2f}, "
    f"WT={INTERNAL_COST_WEIGHT_WT:.2f}, LAI={INTERNAL_COST_WEIGHT_LAI:.2f}, "
    f"balance_penalty={INTERNAL_BALANCE_PENALTY_WEIGHT:.2f}"
)
print(
    f"Knee-selection weights: RR={KNEE_RR_WEIGHT:.2f}, "
    f"Cost={KNEE_COST_WEIGHT:.2f}, Drop={KNEE_DROP_WEIGHT:.2f}, LogBalance={KNEE_BALANCE_LOG_WEIGHT:.2f}, "
    f"ShareBalance={KNEE_BALANCE_SHARE_WEIGHT:.2f}, CapExcess={KNEE_CAP_EXCESS_WEIGHT:.1f}, "
    f"ISAShare={KNEE_ISA_SHARE_WEIGHT:.2f}, LAIShareExcess={KNEE_LAI_SHARE_EXCESS_WEIGHT:.2f}, "
    f"MinRRFraction={KNEE_MIN_RR_FRACTION:.2f}"
)

# Debug mode: set to True to shrink dataset by a factor of 1000 for fast testing
DEBUG_MODE = False


In [ ]:
# ==============================================
# 3) Column name definition (strictly align the feature order of the prediction notebook)
# ==============================================
FEATURE_COLS = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
    "UrbanFrac_2040_2060",
    "PopTotal_2040_2060",
    "ImperviousIndex_2040_2060",
    "LAI_2040_2060",
    "Precip_2040_2060",
    "WTD_2040_2060",
    "HDS_2040_2060",
    "Tas_2040_2060",
    "Huss_2040_2060",
]
SYMBOLS = ["DK", "DB", "DF", "UF", "PT", "IP", "LAI", "PR", "WTD", "HDS", "Tas", "Huss"]
IDX = {s: i for i, s in enumerate(SYMBOLS)}
N_FEATURES = len(FEATURE_COLS)

if len(FEATURE_COLS) != len(SYMBOLS):
    raise ValueError(
        f"FEATURE_COLS length({len(FEATURE_COLS)}) SYMBOLS ({len(SYMBOLS)}) ,."
    )

print("Feature columns:", FEATURE_COLS)
print("SYMBOLS:", SYMBOLS)
print("N_FEATURES:", N_FEATURES)


In [ ]:
# ==================================
# 4) Read in CSV, randomly sample and check columns
# ==================================
df_full = pd.read_csv(CSV_PATH)

need_cols = ["Longitude", "Latitude", "NAME_EN_JX"] + FEATURE_COLS
missing = [c for c in need_cols if c not in df_full.columns]
if missing:
    raise KeyError(f"CSV :{missing}")

df_full = df_full.reset_index().rename(columns={"index": "_source_row_id"})
n_full = int(df_full.shape[0])

if RANDOM_SAMPLE_ENABLED and float(RANDOM_SAMPLE_FRAC) < 1.0:
    sample_n = max(1, int(np.floor(n_full * float(RANDOM_SAMPLE_FRAC))))
    df = (
        df_full.sample(n=sample_n, random_state=int(RANDOM_SAMPLE_SEED), replace=False)
        .sort_values("_source_row_id")
        .reset_index(drop=True)
    )
else:
    sample_n = n_full
    df = df_full.copy().reset_index(drop=True)

SAMPLE_COST_WEIGHT = float(n_full) / float(sample_n)
df["_sample_weight"] = float(SAMPLE_COST_WEIGHT)

X0 = df[FEATURE_COLS].to_numpy(dtype=float)
print("Loaded full CSV:", df_full.shape)
print("Sampled df shape:", df.shape)
print(f"Actual sample ratio: {sample_n}/{n_full} = {sample_n / n_full:.4f}")
print(f"Sample cost weight (n_full / n_sample): {SAMPLE_COST_WEIGHT:.6f}")
print("X0 shape:", X0.shape)
print("NAME_EN_JX sample:", df["NAME_EN_JX"].head().tolist())


In [ ]:
# ==========================================================
# 5) Construct coords: EPSG:4326 -> EPSG:3857, and spell it into (x, y, 0)
# ==========================================================
def build_coords(df: pd.DataFrame) -> np.ndarray:
    """Prioritize geopandas + shapely (consistent with the common writing method of your notebook),
    If the environment does not have geopandas, fallback to pyproj.
    Output coords: (n,3) = (x, y, 0)"""
    if not {"Longitude", "Latitude"}.issubset(df.columns):
        raise KeyError("CSV is missing Longitude / Latitude columns.")

    lon = df["Longitude"].to_numpy(dtype=float)
    lat = df["Latitude"].to_numpy(dtype=float)

    # 1) geopandas path
    try:
        import geopandas as gpd
        from shapely.geometry import Point

        geometry = [Point(xy) for xy in zip(lon, lat)]
        gdf = gpd.GeoDataFrame(df.copy(), geometry=geometry, crs="EPSG:4326").to_crs("EPSG:3857")
        xy = np.array([(p.x, p.y) for p in gdf.geometry], dtype=float)

    except Exception:
        # 2) fallback:pyproj
        try:
            from pyproj import Transformer
            transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
            x, y = transformer.transform(lon, lat)
            xy = np.column_stack([x, y]).astype(float)
        except Exception as e:
            raise RuntimeError(
                "Unable to construct projected coordinates. Please install geopandas (recommended) or pyproj."
            ) from e

    coords = np.hstack([xy, np.zeros((xy.shape[0], 1), dtype=float)])
    return coords

coords = build_coords(df)
print("coords shape:", coords.shape)
print("coords sample:", coords[:3])

In [ ]:
# ===========================================
# 6) pkl, gwr + sigmoid transform_metadata
# ===========================================
import sys
import numpy.core as npcore
import numpy.core.numeric as np_numeric
import numpy.core.multiarray as np_multiarray

sys.modules.setdefault("numpy._core", npcore)
sys.modules.setdefault("numpy._core.numeric", np_numeric)
sys.modules.setdefault("numpy._core.multiarray", np_multiarray)

if not os.path.exists(MODEL_PKL_PATH):
    raise FileNotFoundError(f"GWR model pkl not found: {MODEL_PKL_PATH}")
if not os.path.exists(CLASSIFICATION_ARTIFACTS_PATH):
    raise FileNotFoundError(f"Classification artifacts pkl not found: {CLASSIFICATION_ARTIFACTS_PATH}")
if not os.path.exists(CLASS_BOUNDARIES_PATH):
    raise FileNotFoundError(f"Class boundaries pkl not found: {CLASS_BOUNDARIES_PATH}")

with open(MODEL_PKL_PATH, "rb") as f:
    saved = pickle.load(f)

if "gwr" not in saved:
    raise KeyError("key='gwr' not found in model pkl. Please make sure the save structure is consistent with the notebook.")

gwr = saved["gwr"]

with open(CLASSIFICATION_ARTIFACTS_PATH, "rb") as f:
    gwr_classification_artifacts = pickle.load(f)

with open(CLASS_BOUNDARIES_PATH, "rb") as f:
    class_boundary_data = pickle.load(f)

transform_metadata = gwr_classification_artifacts.get("transform_metadata")
if transform_metadata is None:
    raise KeyError("transform_metadata is missing in gwr_classification_artifacts.pkl to perform unified sigmoid probability mapping.")
class_breaks = np.sort(np.asarray(class_boundary_data.get("class_breaks", []), dtype=float).reshape(-1))
if class_breaks.size < 2 or (not np.isfinite(class_breaks).all()):
    raise KeyError("Class_boundaries.pkl is missing valid class_breaks to perform level reduction optimization.")

print("Loaded gwr model:", type(gwr))
print("Loaded transform_metadata:", transform_metadata)
print("Loaded class_breaks:", class_breaks)


In [ ]:
# ===========================================
# 7) y_pred_gwr, sigmoid (/)
# ===========================================

import os
import contextlib
import numpy as np

def gwr_predict_batch_raw(gwr, coords: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Batch prediction of raw GWR scores: coords=(n,3), X=(n,n_features) -> raw_scores=(n,)"""
    # mgtwr/gwr sometimes calls tqdm to stderr. Here, stderr is redirected to make the output cleaner.
    with open(os.devnull, "w") as fnull, contextlib.redirect_stderr(fnull):
        y_pred, _ = gwr.predict(coords, X)
    return np.asarray(y_pred, dtype=float).reshape(-1)


def gwr_scores_to_probability(raw_scores: np.ndarray, transform_metadata: dict) -> np.ndarray:
    """Map raw GWR scores to 0-1 risk probabilities using the robust sigmoid parameters saved during the training phase."""
    probabilities, _ = gwr_sigmoid_utils.gwr_scores_to_probabilities(
        raw_scores,
        transform_metadata=transform_metadata,
    )
    return np.asarray(probabilities, dtype=float).reshape(-1)


def gwr_predict_batch(gwr, coords: np.ndarray, X: np.ndarray, transform_metadata: dict) -> np.ndarray:
    """Batch predictions and mapped to 0-1 risk probabilities."""
    raw_scores = gwr_predict_batch_raw(gwr, coords, X)
    probabilities = gwr_scores_to_probability(raw_scores, transform_metadata)
    return probabilities.reshape(-1, 1)


y0_raw = gwr_predict_batch_raw(gwr, coords, X0)
y0 = gwr_scores_to_probability(y0_raw, transform_metadata).reshape(-1, 1)
df["_y_pred_base_raw"] = y0_raw
df["_y_pred_base"] = y0.ravel()

print("Base raw GWR-score summary:")
print(pd.Series(y0_raw, name="_y_pred_base_raw").describe())
print("\nBase susceptibility-probability summary (robust sigmoid with saved transform_metadata):")
print(df["_y_pred_base"].describe())


In [ ]:
# =======================================================
# 8) : / topN / ()
# =======================================================

# # Method C: Optimize the specified line number range [START, END)
START, END = 0, 49999   # Change as needed
# START, END = 0, 10 # Change as needed
indices = list(range(START, min(END, len(df))))
# if DEBUG_MODE:
#     indices = indices[:len(indices)//1000]
    
print("Rows to optimize:", len(indices))
print("First 10 indices:", indices[:10])

In [ ]:
# ======================================================
# 9) Define upper and lower limits: fixed variables do not participate in optimization; only IP / LAI / WTD are optimized (according to the current line base ratio)
# ======================================================
# Depaved percentage points
# - Fixed variables do not participate in optimization as decision variables
# - 3 :IP(=ISA), LAI, WTD
# - Use the double constraint of "real relative change range + itemized budget cap" to avoid WTD/LAI cost explosion

OPT_SYMBOLS = ["IP", "LAI", "WTD"]  # Decision variables (3 dimensions)
OPT_IDX10 = [IDX[s] for s in OPT_SYMBOLS]  # Processing step.

# ----Physical/value range (used for clipping to prevent unreasonable negative values)----
ISA_MIN, ISA_MAX = 0.0, 100.0
LAI_MIN, LAI_MAX = 0.0, 543.0
# WTD ; 0, 0 WT .
WTD_MIN, WTD_MAX = -4000.0, 4000.0

def _safe_minmax(a: float, b: float):
    """Avoid upper and lower bound inversion when base may be negative."""
    return (a, b) if a <= b else (b, a)


def _lai_scale_from_base(base_lai_raw: float) -> float:
    """When the original value of LAI is greater than 20, it will be stored and processed as ×100."""
    return 100.0 if float(base_lai_raw) > 20.0 else 1.0


def _wtd_delta_cap_from_budget(cost_cap_usd: float) -> float:
    """Inversely deduce the maximum absolute change d allowed under the budget cap based on the WTD cost function."""
    cost_cap_usd = float(cost_cap_usd)
    if cost_cap_usd <= 0:
        return 0.0

    a = float(C_CAPITAL)
    b = float(C_ENERGY)
    c = -cost_cap_usd / max(float(A_M2), 1e-12)

    if a <= 0:
        if b <= 0:
            return np.inf
        return max(0.0, -c / b)

    disc = b * b - 4.0 * a * c
    disc = max(disc, 0.0)
    return max(0.0, (-b + np.sqrt(disc)) / (2.0 * a))


def _lai_raw_increase_cap_from_budget(base_lai_raw: float, cost_cap_usd: float) -> float:
    """Inversely derive the maximum original value increment allowed under the budget cap based on the LAI cost function."""
    cost_cap_usd = float(cost_cap_usd)
    if cost_cap_usd <= 0:
        return 0.0

    scale = _lai_scale_from_base(base_lai_raw)
    unit_cost_per_actual_lai = float(A_HA) * (float(C_PLANT) + float(T_YEARS) * float(C_MAINT))
    if unit_cost_per_actual_lai <= 0:
        return np.inf

    delta_actual_lai = cost_cap_usd / unit_cost_per_actual_lai
    return max(0.0, delta_actual_lai * scale)


def bounds_from_base(base10: np.ndarray):
    """Based on the current row full-dimensional feature base10, return the lb/ub of 3 controllable variables (IP, LAI, WTD)."""
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    if base10.shape[0] != N_FEATURES:
        raise ValueError(f"base10 must be of length{N_FEATURES}.={base10.shape[0]}")

    lb3 = np.zeros(3, dtype=float)
    ub3 = np.zeros(3, dtype=float)

    # ---- IP/ISA: only reduction allowed; subject to both relative change and budget cap ----
    ip0 = float(np.clip(base10[IDX["IP"]], ISA_MIN, ISA_MAX))
    rel_cap_pts = float(IP_MAX_REDUCTION_FRAC) * ip0

    net_unit_usd_per_m2 = float(C_DEPAVE) - float(T_YEARS) * float(C_FEE)
    if net_unit_usd_per_m2 <= 0:
        budget_cap_pts = rel_cap_pts
    else:
        net_cost_per_point = net_unit_usd_per_m2 * (float(A_M2) / 100.0)
        budget_cap_pts = float(ISA_NET_COST_CAP_USD) / (net_cost_per_point + 1e-12)

    max_reduce_pts = float(np.clip(min(rel_cap_pts, budget_cap_pts), 0.0, ip0))
    lo = float(np.clip(ip0 - max_reduce_pts, ISA_MIN, ISA_MAX))
    hi = float(np.clip(ip0, ISA_MIN, ISA_MAX))
    lb3[0], ub3[0] = (lo, hi) if lo <= hi else (hi, lo)

    # ---- LAI: Real ±30%, and limits the maximum cost corresponding to the "increment" ----
    lai0 = float(np.clip(base10[IDX["LAI"]], LAI_MIN, LAI_MAX))
    lai_rel_lo, lai_rel_hi = _safe_minmax(
        lai0 * (1.0 - float(LAI_REL_CHANGE_FRAC)),
        lai0 * (1.0 + float(LAI_REL_CHANGE_FRAC)),
    )
    lai_up_cap = lai0 + _lai_raw_increase_cap_from_budget(lai0, LAI_NET_COST_CAP_USD)
    lo = float(np.clip(lai_rel_lo, LAI_MIN, LAI_MAX))
    hi = float(np.clip(min(lai_rel_hi, lai_up_cap), LAI_MIN, LAI_MAX))
    if hi < lo:
        hi = lo
    lb3[1], ub3[1] = lo, hi

    # ---- WTD: ±30%, ----
    wtd0 = float(base10[IDX["WTD"]])
    wtd_rel_lo, wtd_rel_hi = _safe_minmax(
        wtd0 * (1.0 - float(WTD_REL_CHANGE_FRAC)),
        wtd0 * (1.0 + float(WTD_REL_CHANGE_FRAC)),
    )
    wtd_delta_cap = _wtd_delta_cap_from_budget(WTD_NET_COST_CAP_USD)
    lo = max(wtd_rel_lo, wtd0 - wtd_delta_cap)
    hi = min(wtd_rel_hi, wtd0 + wtd_delta_cap)
    lo = float(np.clip(lo, WTD_MIN, WTD_MAX))
    hi = float(np.clip(hi, WTD_MIN, WTD_MAX))
    if hi < lo:
        hi = lo
    lb3[2], ub3[2] = lo, hi

    return lb3.tolist(), ub3.tolist()

# Quickly check the upper and lower limits of a point (optional)
test_i = indices[0] if len(indices) else 0
base10 = X0[test_i]
lb3_test, ub3_test = bounds_from_base(base10)

print("Decision vars bounds (based on THIS row base value):")
for j, s in enumerate(OPT_SYMBOLS):
    b = base10[IDX[s]]
    print(f"{s:>3}: lb={lb3_test[j]:.6g}, ub={ub3_test[j]:.6g}, base={b:.6g}")

print(f"WTD delta cap from budget: {_wtd_delta_cap_from_budget(WTD_NET_COST_CAP_USD):.6g} m")
print(f"LAI raw increase cap from budget: {_lai_raw_increase_cap_from_budget(base10[IDX['LAI']], LAI_NET_COST_CAP_USD):.6g}")


In [ ]:
# ======================================================
# 10) NSGA-II: Single-point optimization function (dual objectives: RR maximization + cost minimization)
# ======================================================
# Goal 1 RR: Relative improvement rate of risk probability (the bigger, the better): RR = (y_base - y_opt)/max(y_base, eps)
# Goal 2 Cost: Total economic cost (the smaller, the better): C_total = C_ISA + C_WT + C_LAI

# ---- Cost parameters & spatio-temporal scale ----
# Cost objective and accounting settings
# - T_YEARS: Number of years 2040-2060 (default 20)
# - A_CELL_KM2: unit area (default 1000 km^2; if it is a 10km×10km grid, it should be 100 km^2)
# - COST_MODE: 'absolute' or 'incremental'
# - COST_REPORT: 'total' or 'annual_avg'

A_M2 = float(A_CELL_KM2) * 1e6     # m^2
A_HA = A_M2 / 1e4                 # ha

EPS = 1e-6

import contextlib

def gwr_predict_singlecoord_raw(gwr, coord: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Single point coordinates + multiple sets of X: coord=(3,), X=(N,n_features) -> raw_scores=(N,)"""
    coord = np.asarray(coord, dtype=float).reshape(1, -1)
    coord_rep = np.tile(coord, (X.shape[0], 1))
    with open(os.devnull, "w") as fnull, contextlib.redirect_stderr(fnull):
        y_pred, _ = gwr.predict(coord_rep, X)
    return np.asarray(y_pred, dtype=float).reshape(-1)


def gwr_predict_singlecoord(gwr, coord: np.ndarray, X: np.ndarray, transform_metadata: dict) -> np.ndarray:
    """Single point coordinates + multiple sets of X, returning the mapped 0-1 risk probability."""
    raw_scores = gwr_predict_singlecoord_raw(gwr, coord, X)
    probabilities = gwr_scores_to_probability(raw_scores, transform_metadata)
    return probabilities.reshape(-1, 1)


def assemble_X10_from_X3(base10: np.ndarray, X3: np.ndarray) -> np.ndarray:
    """Put the 3-dimensional decision variables (IP, LAI, WTD) back into full-dimensional features for GWR prediction."""
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    if base10.shape[0] != N_FEATURES:
        raise ValueError(f"base10 must be of length{N_FEATURES}.={base10.shape[0]}")
    base10 = base10.reshape(1, N_FEATURES)
    X3 = np.asarray(X3, dtype=float)
    if X3.ndim == 1:
        X3 = X3.reshape(1, 3)
    if X3.shape[1] != 3:
        raise ValueError("X3 (N,3) , [IP, LAI, WTD].")

    X10 = np.repeat(base10, repeats=X3.shape[0], axis=0)
    X10[:, IDX["IP"]]  = X3[:, 0]
    X10[:, IDX["LAI"]] = X3[:, 1]
    X10[:, IDX["WTD"]] = X3[:, 2]
    return X10


def _lai_scale_from_base(base_lai_raw: float) -> float:
    """The LAI range of your data is 400+, probably ×100; a scale judgment is automatically made here."""
    return 100.0 if base_lai_raw > 20.0 else 1.0


def cost_components(base10: np.ndarray, X3: np.ndarray):
    """Calculate (C_ISA, C_WT, C_LAI, C_total) according to the cost function you gave, and return an array of shape (N,1).

    COST_MODE:
      - 'absolute' : "Total cost" in the output scenario (including annual costs/maintenance of existing ISA/LAI)
      - 'incremental': Outputs the "incremental cost of governance" relative to the baseline (closer to the governance investment/budget caliber; recommended)
        * ISA: depave one-time cost + (fee_opt - fee_base) 20-year difference
        * LAI: plant one-time cost + (maint_opt - maint_base) 20-year difference
        * WT: Originally the cost based on the variation d (incremental caliber)

    COST_REPORT:
      - 'total' : 20 years cumulative (consistent with disaster time scale)
      - 'annual_avg' : annual average (= total / T_YEARS) for easy comparison of "annual costs""""
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    X3 = np.asarray(X3, dtype=float)
    if X3.ndim == 1:
        X3 = X3.reshape(1, 3)

    isa_orig = float(base10[IDX["IP"]])
    wt_orig  = float(base10[IDX["WTD"]])
    lai_orig_raw = float(base10[IDX["LAI"]])

    isa_opt = X3[:, 0]
    lai_opt_raw = X3[:, 1]
    wt_opt  = X3[:, 2]

    # ---- ISA Cost (IP is a scale of 0-100)----
    isa_orig_c = float(np.clip(isa_orig, 0.0, 100.0))
    isa_opt_c  = np.clip(isa_opt,  0.0, 100.0)

    # ISA reduction cost
    depave_pts = np.maximum(0.0, isa_orig_c - isa_opt_c)  # Depaved percentage points
    C_ISA_depave = A_M2 * float(C_DEPAVE) * (depave_pts / 100.0)

    # Annual fee: Proportional to the ISA area
    fee_base = A_M2 * float(T_YEARS) * float(C_FEE) * (isa_orig_c / 100.0)
    fee_opt  = A_M2 * float(T_YEARS) * float(C_FEE) * (isa_opt_c  / 100.0)

    if str(COST_MODE).lower() == "incremental":
        # Incremental caliber: only look at the difference relative to the baseline (reducing ISA will bring negative costs = savings)
        C_ISA_fee = fee_opt - fee_base
    else:
        # Absolute caliber: directly use the total fee in the scenario
        C_ISA_fee = fee_opt

    C_ISA = C_ISA_depave + C_ISA_fee

    # ---- WT ( d)----
    d = np.abs(wt_opt - wt_orig)
    C_WT = A_M2 * (float(C_ENERGY) * d + float(C_CAPITAL) * d**2)

    # ---- LAI ( /100)----
    scale = _lai_scale_from_base(lai_orig_raw)
    lai_orig = max(lai_orig_raw / scale, 0.0)
    lai_opt  = np.maximum(lai_opt_raw / scale, 0.0)

    # Planting cost: Only one-time cost will be added to LAI
    delta_lai_plus = np.maximum(0.0, lai_opt - lai_orig)
    C_LAI_plant = A_HA * float(C_PLANT) * delta_lai_plus

    # Maintenance cost: proportional to LAI level (annual)
    maint_base = A_HA * float(T_YEARS) * float(C_MAINT) * lai_orig
    maint_opt  = A_HA * float(T_YEARS) * float(C_MAINT) * lai_opt

    if str(COST_MODE).lower() == "incremental":
        C_LAI_maint = maint_opt - maint_base
    else:
        C_LAI_maint = maint_opt

    C_LAI = C_LAI_plant + C_LAI_maint
    # ---- Force the cost to be non-negative (very important: cost in the objective function must >= 0)----
    # Note: Under the incremental caliber, some balance items may be negative (representing "savings"), but as "expenditure" the target value should not be negative.
    # Therefore, each sub-item and the total cost are truncated: cost = max(cost, 0)
    C_ISA = np.maximum(C_ISA, 0.0)
    C_WT  = np.maximum(C_WT,  0.0)
    C_LAI = np.maximum(C_LAI, 0.0)

    C_total = C_ISA + C_WT + C_LAI
    # Reporting caliber: annual average or 20-year cumulative
    if str(COST_REPORT).lower() == "annual_avg":
        C_ISA = C_ISA / float(T_YEARS)
        C_WT  = C_WT  / float(T_YEARS)
        C_LAI = C_LAI / float(T_YEARS)
        C_total = C_total / float(T_YEARS)

    return (np.asarray(C_ISA).reshape(-1,1),
            np.asarray(C_WT).reshape(-1,1),
            np.asarray(C_LAI).reshape(-1,1),
            np.asarray(C_total).reshape(-1,1))


def component_cap_excess_ratio(cost_parts: np.ndarray) -> np.ndarray:
    """cap ; 0 ."""
    cost_parts = np.maximum(np.asarray(cost_parts, dtype=float), 0.0)
    if cost_parts.ndim == 1:
        cost_parts = cost_parts.reshape(1, -1)
    caps = np.array([
        max(float(ISA_NET_COST_CAP_USD), EPS),
        max(float(WTD_NET_COST_CAP_USD), EPS),
        max(float(LAI_NET_COST_CAP_USD), EPS),
    ], dtype=float).reshape(1, 3)
    excess = np.maximum(cost_parts - caps, 0.0) / caps
    return np.sum(excess, axis=1)


def optimization_cost_objective(C_ISA: np.ndarray, C_WT: np.ndarray, C_LAI: np.ndarray) -> np.ndarray:
    """Internal optimization goal: do not change the unit price, only use soft weights and imbalance penalties,
    To prevent ISA from being pressured by the 10km grid cost scale of approximately 20 million USD per 1pct for a long time and barely participating.
    At the same time, a strong hard penalty is imposed on candidate solutions that exceed the item budget cap."""
    C_ISA = np.asarray(C_ISA, dtype=float).reshape(-1, 1)
    C_WT  = np.asarray(C_WT, dtype=float).reshape(-1, 1)
    C_LAI = np.asarray(C_LAI, dtype=float).reshape(-1, 1)

    weighted_total = (
        float(INTERNAL_COST_WEIGHT_ISA) * C_ISA +
        float(INTERNAL_COST_WEIGHT_WT)  * C_WT +
        float(INTERNAL_COST_WEIGHT_LAI) * C_LAI
    )

    cost_stack = np.hstack([C_ISA, C_WT, C_LAI])
    cost_mean = np.mean(cost_stack, axis=1, keepdims=True)
    imbalance = np.sqrt(np.mean((cost_stack - cost_mean) ** 2, axis=1, keepdims=True))
    cap_excess = component_cap_excess_ratio(cost_stack).reshape(-1, 1)
    hard_penalty = 1.0e9 * cap_excess

    return weighted_total + float(INTERNAL_BALANCE_PENALTY_WEIGHT) * imbalance + hard_penalty


def probability_to_risk_level(probabilities: np.ndarray) -> np.ndarray:
    """Use the natural breakpoints saved during the training phase to map the 0-1 probability to 1-5 risk levels."""
    probs = np.asarray(probabilities, dtype=float).reshape(-1)
    breaks = np.asarray(class_breaks, dtype=float).reshape(-1)
    probs = np.clip(probs, float(breaks[0]), float(breaks[-1]))
    return np.digitize(probs, breaks[1:-1], right=True) + 1


def risk_metrics(base_y: float, y_opt: np.ndarray):
    """RR,  drop, utility."""
    y_opt = np.asarray(y_opt, dtype=float).reshape(-1)
    denom = max(float(base_y), EPS)
    RR = np.clip((float(base_y) - y_opt) / denom, 0.0, 1.0)

    base_level = int(probability_to_risk_level(np.array([base_y]))[0])
    opt_level = probability_to_risk_level(y_opt)
    drop = np.clip(base_level - opt_level, 0, class_breaks.size - 2).astype(float)

    drop_norm = drop / max(float(class_breaks.size - 2), 1.0)
    utility = (
        RR
        + float(RISK_LEVEL_DROP_OBJECTIVE_WEIGHT) * drop_norm
        + float(RISK_LEVEL_2PLUS_OBJECTIVE_BONUS) * (drop >= 2.0).astype(float)
        + float(RISK_LEVEL_3PLUS_OBJECTIVE_BONUS) * (drop >= 3.0).astype(float)
    )
    return RR.reshape(-1, 1), drop.reshape(-1, 1), utility.reshape(-1, 1)


def objectives(base_y: float, y_opt: np.ndarray, C_obj: np.ndarray):
    """Construct two targets ObjV = [Level-aware utility, Cost].
    Utility contains both continuous RR and natural breakpoint level reduction bonuses, directly pushing 2/3 levels down."""
    _, _, utility = risk_metrics(base_y, y_opt)
    return np.hstack([utility, np.asarray(C_obj).reshape(-1,1)])


def component_log_balance_penalty(cost_parts: np.ndarray) -> np.ndarray:
    """Measure whether the three types of costs are of similar magnitude; the smaller the cost, the more balanced it is."""
    cost_parts = np.maximum(np.asarray(cost_parts, dtype=float), 0.0)
    if cost_parts.ndim == 1:
        cost_parts = cost_parts.reshape(1, -1)
    log_costs = np.log1p(cost_parts)
    return np.max(log_costs, axis=1) - np.min(log_costs, axis=1)


def component_share_balance_penalty(cost_parts: np.ndarray) -> np.ndarray:
    """1/3 : 1/3 : 1/3;."""
    cost_parts = np.maximum(np.asarray(cost_parts, dtype=float), 0.0)
    if cost_parts.ndim == 1:
        cost_parts = cost_parts.reshape(1, -1)
    total = np.sum(cost_parts, axis=1, keepdims=True)
    shares = np.divide(cost_parts, total, out=np.zeros_like(cost_parts), where=total > EPS)
    return np.mean((shares - (1.0 / 3.0)) ** 2, axis=1)


def pick_knee_solution(ObjV: np.ndarray, cost_parts=None):
    """: 1) RR Pareto RR KNEE_MIN_RR_FRACTION ; 2) RR ,,, ,  cap; 3) ObjV 3/4, utility / drop, bal8 ."""
    rr = np.asarray(ObjV[:, 0], dtype=float)
    cost = np.asarray(ObjV[:, 1], dtype=float)
    idx_all = np.arange(rr.shape[0])

    rr_max_all = float(np.nanmax(rr)) if rr.size else 0.0
    min_rr_fraction = float(globals().get("KNEE_MIN_RR_FRACTION", 0.0))
    if rr_max_all > EPS and min_rr_fraction > 0.0:
        candidate_mask = rr >= (min_rr_fraction * rr_max_all)
        if not np.any(candidate_mask):
            candidate_mask = np.ones_like(rr, dtype=bool)
    else:
        candidate_mask = np.ones_like(rr, dtype=bool)

    drop = np.asarray(ObjV[:, 3], dtype=float) if ObjV.shape[1] >= 4 else np.zeros_like(rr)
    rr_c = rr[candidate_mask]
    cost_c = cost[candidate_mask]
    drop_c = drop[candidate_mask]
    idx_c = idx_all[candidate_mask]

    rr_min, rr_max = float(np.min(rr_c)), float(np.max(rr_c))
    c_min, c_max   = float(np.min(cost_c)), float(np.max(cost_c))

    rr_norm = (rr_c - rr_min) / (rr_max - rr_min + EPS)
    c_norm  = (cost_c - c_min) / (c_max - c_min + EPS)

    drop_max = float(np.max(drop_c)) if drop_c.size else 0.0
    drop_norm = drop_c / max(drop_max, 1.0)
    score = (
        float(KNEE_RR_WEIGHT) * (1.0 - rr_norm) ** 2
        + float(KNEE_COST_WEIGHT) * (c_norm) ** 2
        - float(KNEE_DROP_WEIGHT) * drop_norm
    )

    if cost_parts is not None:
        cost_parts_c = np.asarray(cost_parts, dtype=float)[candidate_mask]
        log_bal = component_log_balance_penalty(cost_parts_c)
        share_bal = component_share_balance_penalty(cost_parts_c)
        cap_excess = component_cap_excess_ratio(cost_parts_c)
        total_parts = np.sum(np.maximum(cost_parts_c, 0.0), axis=1, keepdims=True)
        cost_shares = np.divide(
            np.maximum(cost_parts_c, 0.0),
            total_parts,
            out=np.zeros_like(cost_parts_c, dtype=float),
            where=total_parts > EPS,
        )
        isa_share = cost_shares[:, 0]
        lai_share = cost_shares[:, 2]

        lb_min, lb_max = float(np.min(log_bal)), float(np.max(log_bal))
        if lb_max > lb_min:
            lb_norm = (log_bal - lb_min) / (lb_max - lb_min + EPS)
            score = score + float(KNEE_BALANCE_LOG_WEIGHT) * (lb_norm ** 2)

        sb_min, sb_max = float(np.min(share_bal)), float(np.max(share_bal))
        if sb_max > sb_min:
            sb_norm = (share_bal - sb_min) / (sb_max - sb_min + EPS)
            score = score + float(KNEE_BALANCE_SHARE_WEIGHT) * (sb_norm ** 2)

        score = score + float(KNEE_CAP_EXCESS_WEIGHT) * cap_excess
        score = score - float(KNEE_ISA_SHARE_WEIGHT) * isa_share
        score = score + float(KNEE_LAI_SHARE_EXCESS_WEIGHT) * np.maximum(lai_share - 0.45, 0.0) ** 2

    return int(idx_c[int(np.argmin(score))])

def optimize_one_point(
    gwr,
    coord: np.ndarray,
    base10: np.ndarray,
    base_y: float,
    transform_metadata: dict,
    nind: int = 60,
    maxgen: int = 80,
    seed: int = 1,
    pm: float = 0.2,
    xover: float = 0.7,
):
    """Do NSGA-II bi-objective optimization on a single point:
      - Goal 1: Maximize RR (the bigger the risk improvement, the better)
      - Goal 2: Cost minimization (internally a balanced soft cost target)
    Decision dimension = 3 (IP, LAI, WTD), the remaining fixed variables maintain base values, but still participate in prediction.

    Return:
        best_var10 (n_features,),
        best_RR (float),
        best_Cost (float),
        Vars3 (k,3),
        ObjV_actual (k,2) # [RR, actual_total_cost]"""
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    lb3, ub3 = bounds_from_base(base10)

    class GWRProblem(ea.Problem):
        def __init__(self):
            name = "GWR_RR_Max_Cost_Min"
            M = 2
            maxormins = [-1, 1]     # RR maximizes; Cost minimizes
            Dim = 3
            varTypes = [0] * Dim
            lbin = [1] * Dim
            ubin = [1] * Dim
            ea.Problem.__init__(self, name, M, maxormins, Dim, varTypes, lb3, ub3, lbin, ubin)

        def evalVars(self, X3):
            X10 = assemble_X10_from_X3(base10, X3)
            y_opt = gwr_predict_singlecoord(gwr, coord, X10, transform_metadata)
            C_ISA, C_WT, C_LAI, _ = cost_components(base10, X3)
            C_obj = optimization_cost_objective(C_ISA, C_WT, C_LAI)
            ObjV = objectives(base_y, y_opt, C_obj)
            return ObjV

    problem = GWRProblem()
    pop = ea.Population(Encoding="RI", NIND=nind)
    algorithm = ea.moea_NSGA2_templet(problem, pop, MAXGEN=maxgen, logTras=0)

    algorithm.mutOper.Pm = pm
    algorithm.recOper.XOVR = xover

    res = ea.optimize(
        algorithm,
        seed=seed,
        verbose=False,
        drawing=0,
        outputMsg=False,
        saveFlag=False,
        drawLog=False,
    )

    Vars3 = res["Vars"]
    ObjV_internal = res["ObjV"]

    X10_all = assemble_X10_from_X3(base10, Vars3)
    y_opt_all = gwr_predict_singlecoord(gwr, coord, X10_all, transform_metadata).reshape(-1)
    RR_all, drop_all, utility_all = risk_metrics(base_y, y_opt_all)
    C_ISA_all, C_WT_all, C_LAI_all, C_total_all = cost_components(base10, Vars3)
    cost_parts = np.hstack([C_ISA_all, C_WT_all, C_LAI_all])
    # ObjV_actual: [real RR, real total cost, level-aware utility, level reduction drop]
    ObjV_actual = np.hstack([RR_all, C_total_all, utility_all, drop_all])

    best_i = pick_knee_solution(ObjV_actual, cost_parts)
    best_var3 = Vars3[best_i, :]
    best_var10 = assemble_X10_from_X3(base10, best_var3).reshape(-1)

    best_RR = float(ObjV_actual[best_i, 0])
    best_Cost = float(np.maximum(C_total_all[best_i, 0], 0.0))

    return best_var10, best_RR, best_Cost, Vars3, ObjV_actual

print("Balanced multi-objective optimization function ready.")


In [ ]:
# ======================================================
# 11) Perform optimization (point by point)
# Output: summary (a compromise solution for each point) + pareto (a long list of Pareto solution sets for each point)
# Newly added: breakpoint resume + batch download to reduce frequent hard disk read and write
# ======================================================
import gc
import time

FIXED_SYMBOLS = ["DK", "DB", "DF", "UF", "PT", "PR", "HDS", "Tas", "Huss"]  # 9 fixed variables, the order is aligned with the prediction notebook
FIXED_IDX10 = [IDX[s] for s in FIXED_SYMBOLS]

def _read_ckpt_csv(path: str) -> pd.DataFrame:
    if (not os.path.exists(path)) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, on_bad_lines="skip")
    except TypeError:
        # The old version of pandas does not have on_bad_lines
        return pd.read_csv(path)
    except Exception as e:
        print(f"Warning: failed to read checkpoint {path}: {e}")
        return pd.DataFrame()

def _append_df_safely(df_part: pd.DataFrame, path: str):
    """Write CSV in append mode. By default, it only flushes and does not fsync every time to avoid frequent hard disk synchronization."""
    if df_part is None or len(df_part) == 0:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    need_header = (not os.path.exists(path)) or (os.path.getsize(path) == 0)
    with open(path, "a", encoding="utf-8-sig", newline="") as f:
        df_part.to_csv(f, index=False, header=need_header)
        f.flush()
        if FSYNC_EVERY_SAVE:
            os.fsync(f.fileno())

# ---- Load the existing checkpoint to realize breakpoint continuation ----
summary_ckpt_df = _read_ckpt_csv(CHECKPOINT_SUMMARY_PATH)
pareto_ckpt_df  = _read_ckpt_csv(CHECKPOINT_PARETO_PATH)

if "row_id" in summary_ckpt_df.columns:
    summary_ckpt_df = summary_ckpt_df.drop_duplicates(subset=["row_id"], keep="last")
    done_set = set(summary_ckpt_df["row_id"].astype(int).tolist())
else:
    done_set = set()

if ("row_id" in pareto_ckpt_df.columns) and ("sol_id" in pareto_ckpt_df.columns):
    pareto_ckpt_df = pareto_ckpt_df.drop_duplicates(subset=["row_id", "sol_id"], keep="last")

if RESUME_FROM_CHECKPOINT:
    todo_indices = [int(i) for i in indices if int(i) not in done_set]
else:
    todo_indices = [int(i) for i in indices]

print(f"Checkpoint summary: {CHECKPOINT_SUMMARY_PATH}")
print(f"Checkpoint pareto : {CHECKPOINT_PARETO_PATH}")
print(f"Already finished : {len(done_set)} rows")
print(f"Rows to run now  : {len(todo_indices)} / {len(indices)}")

summary_buffer = []
pareto_buffer = []

for run_k, i in enumerate(todo_indices, start=1):
    base10 = X0[i, :]
    coord = coords[i, :]
    name_en_jx = str(df.loc[i, "NAME_EN_JX"])
    base_y = float(df.loc[i, "_y_pred_base"])   # has been mapped to [0,1] risk probability in block 7

    # NaN / inf
    if np.any(~np.isfinite(base10)) or np.any(~np.isfinite(coord)) or (not np.isfinite(base_y)):
        summary_row = {
            "row_id": i,
            "source_row_id": int(df.loc[i, "_source_row_id"]),
            "Longitude": float(df.loc[i, "Longitude"]),
            "Latitude": float(df.loc[i, "Latitude"]),
            "NAME_EN_JX": name_en_jx,
            "y_pred_base": base_y,
            "RR_best": np.nan,
            "Cost_best": np.nan,
            "y_pred_opt_best": np.nan,
            "sample_weight": float(df.loc[i, "_sample_weight"]),
            "status": "skip_nan",
        }
        summary_buffer.append(summary_row)

        if len(summary_buffer) >= int(SAVE_EVERY_ROWS):
            _append_df_safely(pd.DataFrame(summary_buffer), CHECKPOINT_SUMMARY_PATH)
            summary_buffer = []
        continue

    # Key: Each row uses a fixed and reproducible seed to facilitate stable results after resuming the run after a breakpoint.
    row_seed = int(SEED) + int(i)

    best_var10, best_RR, best_Cost, Vars3, ObjV = optimize_one_point(
        gwr=gwr,
        coord=coord,
        base10=base10,
        base_y=base_y,
        transform_metadata=transform_metadata,
        nind=NIND,
        maxgen=MAXGEN,
        seed=row_seed,
        pm=PM,
        xover=XOVER,
    )

    # Calculate the true risk probability prediction corresponding to the Pareto solution again (batch, mapped to [0,1] through sigmoid), to avoid the distortion of the formula inversion when RR=0
    X10_pareto = assemble_X10_from_X3(base10, Vars3)
    y_opt_all = gwr_predict_singlecoord(gwr, coord, X10_pareto, transform_metadata).reshape(-1)

    # , summary / pareto
    C_ISA_all, C_WT_all, C_LAI_all, C_total_all = cost_components(base10, Vars3)
    cost_parts = np.hstack([C_ISA_all, C_WT_all, C_LAI_all])

    # Internally consistent with optimize_one_point: find another compromise solution index
    best_i = pick_knee_solution(ObjV, cost_parts)
    y_opt_best = float(y_opt_all[best_i])   # is [0,1] risk probability

    # Recompute RR in the [0, 1] range
    denom = max(base_y, EPS)
    best_RR = float(np.clip((base_y - y_opt_best) / denom, 0.0, 1.0))
    best_Cost = float(np.maximum(C_total_all[best_i, 0], 0.0))

    # Fixed variable self-test (theoretically it should be 0)
    fixed_max_abs_diff = float(np.max(np.abs(best_var10[FIXED_IDX10] - base10[FIXED_IDX10])))

    # 3
    IP_best  = float(best_var10[IDX["IP"]])
    LAI_best = float(best_var10[IDX["LAI"]])
    WTD_best = float(best_var10[IDX["WTD"]])

    # Cost breakdown of the compromise solution
    C_ISA = float(C_ISA_all[best_i, 0])
    C_WT = float(C_WT_all[best_i, 0])
    C_LAI = float(C_LAI_all[best_i, 0])
    C_total = float(C_total_all[best_i, 0])

    summary_row = {
        "row_id": i,
        "source_row_id": int(df.loc[i, "_source_row_id"]),
        "Longitude": float(df.loc[i, "Longitude"]),
        "Latitude": float(df.loc[i, "Latitude"]),
        "NAME_EN_JX": name_en_jx,
        "y_pred_base": float(np.clip(base_y, 0.0, 1.0)),
        "y_pred_opt_best": float(np.clip(y_opt_best, 0.0, 1.0)),
        "RR_best": best_RR,                  # [0,1]
        "Cost_best": best_Cost,              # Forced >=0
        "Cost_ISA": max(C_ISA, 0.0),
        "Cost_WT": max(C_WT, 0.0),
        "Cost_LAI": max(C_LAI, 0.0),
        "Cost_total_check": max(C_total, 0.0),

        "IP_opt": IP_best,
        "LAI_opt": LAI_best,
        "WTD_opt": WTD_best,

        "IP_change_pct": (IP_best / float(base10[IDX["IP"]]) - 1.0) if base10[IDX["IP"]] != 0 else np.nan,
        "LAI_change_pct": (LAI_best / float(base10[IDX["LAI"]]) - 1.0) if base10[IDX["LAI"]] != 0 else np.nan,
        "WTD_change_pct": (WTD_best / float(base10[IDX["WTD"]]) - 1.0) if base10[IDX["WTD"]] != 0 else np.nan,

        "fixed_max_abs_diff": fixed_max_abs_diff,
        "pareto_size": int(ObjV.shape[0]),
        "row_seed": row_seed,
        "sample_weight": float(df.loc[i, "_sample_weight"]),
        "status": "ok",
    }
    summary_buffer.append(summary_row)

    # ---- Save Pareto solution set (long table) ----
    pareto_part = []
    for j in range(ObjV.shape[0]):
        rr_j = float(np.clip(ObjV[j, 0], 0.0, 1.0))
        cost_j = float(np.maximum(ObjV[j, 1], 0.0))
        y_opt_j = float(np.clip(y_opt_all[j], 0.0, 1.0))

        ip_j, lai_j, wtd_j = float(Vars3[j,0]), float(Vars3[j,1]), float(Vars3[j,2])

        # Cost breakdown (reuse the vectorized results above)
        C_ISA_j = float(C_ISA_all[j, 0])
        C_WT_j = float(C_WT_all[j, 0])
        C_LAI_j = float(C_LAI_all[j, 0])
        C_total_j = float(C_total_all[j, 0])

        pareto_part.append({
            "row_id": i,
            "source_row_id": int(df.loc[i, "_source_row_id"]),
            "sol_id": j,
            "Longitude": float(df.loc[i, "Longitude"]),
            "Latitude": float(df.loc[i, "Latitude"]),
            "NAME_EN_JX": name_en_jx,
            "y_pred_base": float(np.clip(base_y, 0.0, 1.0)),
            "y_pred_opt": y_opt_j,
            "RR": rr_j,                       # [0,1]
            "Cost_total": max(C_total_j, 0.0),  # Forced >=0
            "Cost_ISA": max(C_ISA_j, 0.0),
            "Cost_WT": max(C_WT_j, 0.0),
            "Cost_LAI": max(C_LAI_j, 0.0),

            "IP_opt": ip_j,
            "LAI_opt": lai_j,
            "WTD_opt": wtd_j,
            "row_seed": row_seed,
            "sample_weight": float(df.loc[i, "_sample_weight"]),
        })

    pareto_buffer.extend(pareto_part)

    # ---- Periodic placement ----
    if (len(summary_buffer) >= int(SAVE_EVERY_ROWS)) or (run_k == len(todo_indices)):
        # Write pareto first, then summary; summary is regarded as a mark of "the row is completed"
        if len(pareto_buffer) > 0:
            _append_df_safely(pd.DataFrame(pareto_buffer), CHECKPOINT_PARETO_PATH)
            pareto_buffer = []

        if len(summary_buffer) > 0:
            _append_df_safely(pd.DataFrame(summary_buffer), CHECKPOINT_SUMMARY_PATH)
            summary_buffer = []

        gc.collect()

    if (run_k % 10 == 0) or (run_k == len(todo_indices)):
        print(
            f"[{run_k}/{len(todo_indices)}] row_id={i} "
            f"base={base_y:.6f} y_opt={y_opt_best:.6f} RR_best={best_RR:.4f} "
            f"Cost_best={best_Cost:.3e} pareto={ObjV.shape[0]}"
        )

def _scale_costs_for_export(df_in: pd.DataFrame) -> pd.DataFrame:
    """,."""
    return df_in.copy()

summary_df = _read_ckpt_csv(CHECKPOINT_SUMMARY_PATH)
pareto_df = _read_ckpt_csv(CHECKPOINT_PARETO_PATH)

if "row_id" in summary_df.columns:
    summary_df = summary_df.drop_duplicates(subset=["row_id"], keep="last").sort_values("row_id").reset_index(drop=True)

if ("row_id" in pareto_df.columns) and ("sol_id" in pareto_df.columns):
    pareto_df = pareto_df.drop_duplicates(subset=["row_id", "sol_id"], keep="last").sort_values(["row_id", "sol_id"]).reset_index(drop=True)

summary_df = _scale_costs_for_export(summary_df)
pareto_df = _scale_costs_for_export(pareto_df)

summary_df.to_csv(FINAL_SUMMARY_PATH, index=False, encoding="utf-8-sig")
pareto_df.to_csv(FINAL_PARETO_PATH, index=False, encoding="utf-8-sig")

print("summary_df:", summary_df.shape)
print("pareto_df :", pareto_df.shape)

display(summary_df.head())

print("\nCost summary (export values, no scaling):")
summary_cols = [c for c in ["Cost_best", "Cost_ISA", "Cost_WT", "Cost_LAI", "Cost_total_check"] if c in summary_df.columns]
if summary_cols:
    display(summary_df[summary_cols].describe(percentiles=[0.5, 0.9, 0.99]).T)
